# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../02_activities/documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

# join the page into one string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

# checked the loaded document
print(len(docs))
print(document_text[:100])


13
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with t


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from openai import OpenAI
from pydantic import BaseModel, Field
from IPython.display import display, Markdown
import os

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

class PDFsummary(BaseModel):
    Author: str = Field(description="Author of the article")
    Title: str = Field(description= "Title of the article")
    Relevance: str = Field(description= "a statement, no longer than one paragraph, that explains why is this article relevant " \
    "for an AI professional in their professional development.")
    Summary: str = Field(description="A concise summary of the article, no longer than 1000 tokens.")
    Tone: str = Field(description="The specific tone used to write the summary.")
    InputTokens: int
    OutputTokens: int


In [4]:
tone = "Formal Academic Writing"

developer_instructions = f"""
You are an expert summarization assistant.

Your task is to analyze an article and return a structured response.

Requirements:
- Use the tone: {tone}
- Write the summary in a clear and distinguishable version of that tone.
- Keep the summary concise and under 1000 tokens.
- The relevance section must be no longer than one paragraph.
- Do not invent information that is not present in the article.
- Set InputTokens and OutputTokens to 0. These values will be filled programmatically after generation.
"""

user_prompt = f"""
Please summarize the following article.

<context>
{document_text}
</context>
"""

In [5]:
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": developer_instructions,},
        {"role": "user", "content": user_prompt,},
    ],
    text_format=PDFsummary,
)

display(Markdown(response.output_text))

{"Author":"Peter F. Drucker","Title":"Managing Oneself","Relevance":"This article is highly relevant for AI professionals as it emphasizes the necessity of self-awareness, personal strengths, and adaptability in the rapidly evolving knowledge economy. Understanding one’s abilities and values is crucial to navigate career developments and make meaningful contributions in the AI field, which is increasingly reliant on individual initiative and expertise.","Summary":"In the article 'Managing Oneself', Peter F. Drucker articulates the importance of self-management and personal accountability in the knowledge economy. He posits that successful individuals must act as their own chief executive officers, taking charge of their careers in light of diminished corporate career management. To do so effectively, one must possess a deep understanding of their strengths, weaknesses, preferred work styles, values, and environments in which they excel. Drucker introduces feedback analysis as a tool to help identify strengths by comparing expected and actual outcomes of decisions over time. He emphasizes the importance of recognizing how one learns and performs, whether through listening or reading, and adapting accordingly. Additionally, he discusses the significance of aligning with organizational values to avoid frustration and inefficiency. The article advocates for proactive engagement in one’s career development by determining where one can contribute effectively and acknowledging it is crucial to manage relationships responsibly. Drucker concludes by encouraging individuals to prepare for the second half of life by cultivating new interests or parallel careers, as well as clarifying their values, thus promoting ongoing personal and professional growth throughout one’s career.","Tone":"Formal Academic Writing","InputTokens":0,"OutputTokens":0}

In [6]:
pdf_summary = response.output_parsed

pdf_summary = pdf_summary.model_copy(
    update={
        "InputTokens": response.usage.input_tokens,
        "OutputTokens": response.usage.output_tokens,
    }
)

pdf_summary

PDFsummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance='This article is highly relevant for AI professionals as it emphasizes the necessity of self-awareness, personal strengths, and adaptability in the rapidly evolving knowledge economy. Understanding one’s abilities and values is crucial to navigate career developments and make meaningful contributions in the AI field, which is increasingly reliant on individual initiative and expertise.', Summary="In the article 'Managing Oneself', Peter F. Drucker articulates the importance of self-management and personal accountability in the knowledge economy. He posits that successful individuals must act as their own chief executive officers, taking charge of their careers in light of diminished corporate career management. To do so effectively, one must possess a deep understanding of their strengths, weaknesses, preferred work styles, values, and environments in which they excel. Drucker introduces feedback analysis as a to

In [7]:
print(pdf_summary.Title)
print(pdf_summary.Author)
print(pdf_summary.Relevance)
print(pdf_summary.Summary)
print(pdf_summary.Tone)
print(pdf_summary.InputTokens)
print(pdf_summary.OutputTokens)

Managing Oneself
Peter F. Drucker
This article is highly relevant for AI professionals as it emphasizes the necessity of self-awareness, personal strengths, and adaptability in the rapidly evolving knowledge economy. Understanding one’s abilities and values is crucial to navigate career developments and make meaningful contributions in the AI field, which is increasingly reliant on individual initiative and expertise.
In the article 'Managing Oneself', Peter F. Drucker articulates the importance of self-management and personal accountability in the knowledge economy. He posits that successful individuals must act as their own chief executive officers, taking charge of their careers in light of diminished corporate career management. To do so effectively, one must possess a deep understanding of their strengths, weaknesses, preferred work styles, values, and environments in which they excel. Drucker introduces feedback analysis as a tool to help identify strengths by comparing expected 

In [8]:
display(Markdown(pdf_summary.Summary))

In the article 'Managing Oneself', Peter F. Drucker articulates the importance of self-management and personal accountability in the knowledge economy. He posits that successful individuals must act as their own chief executive officers, taking charge of their careers in light of diminished corporate career management. To do so effectively, one must possess a deep understanding of their strengths, weaknesses, preferred work styles, values, and environments in which they excel. Drucker introduces feedback analysis as a tool to help identify strengths by comparing expected and actual outcomes of decisions over time. He emphasizes the importance of recognizing how one learns and performs, whether through listening or reading, and adapting accordingly. Additionally, he discusses the significance of aligning with organizational values to avoid frustration and inefficiency. The article advocates for proactive engagement in one’s career development by determining where one can contribute effectively and acknowledging it is crucial to manage relationships responsibly. Drucker concludes by encouraging individuals to prepare for the second half of life by cultivating new interests or parallel careers, as well as clarifying their values, thus promoting ongoing personal and professional growth throughout one’s career.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

The cell below sets up a single judge model (`gpt-4o-mini` routed through the class API gateway) and defines the four required metrics:

1. **`SummarizationMetric`** with five bespoke assessment questions drawn from the article.
2. **`GEval` — Coherence**, with five evaluation steps targeting logical flow and clarity.
3. **`GEval` — Tonality**, with five evaluation steps targeting the requested tone (`Formal Academic Writing`).
4. **`GEval` — Safety**, with five evaluation steps targeting harmful, biased, or unsafe content.

All four are wrapped in a helper function `evaluate_summary(source_text, summary_text)` that returns a Pydantic `SummaryEvaluation` object with the eight required fields (`SummarizationScore` / `SummarizationReason`, and the same pair for Coherence, Tonality, Safety). Wrapping it means the Enhancement section can call the exact same evaluator on the revised summary with one line.

In [ ]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel

# Judge model — single instance reused by all four metrics.
judge_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

# 1) SummarizationMetric — five bespoke yes/no assessment questions drawn from the article.
summarization_questions = [
    "Does the summary state that knowledge workers must take responsibility for managing their own careers?",
    "Does the summary mention feedback analysis as a way to identify one's strengths?",
    "Does the summary discuss understanding how one performs best (e.g., reader vs. listener, learning style)?",
    "Does the summary reference the importance of identifying one's values and where one belongs?",
    "Does the summary mention preparing for the second half of life through a second or parallel career?",
]

summarization_metric = SummarizationMetric(
    threshold=0.7,
    model=judge_model,
    assessment_questions=summarization_questions,
    include_reason=True,
)

# 2) G-Eval: Coherence — five evaluation steps.
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Check that the summary has a clear, logical flow from beginning to end.",
        "Verify that sentences connect smoothly with natural transitions between ideas.",
        "Penalize abrupt topic shifts, non-sequiturs, or disconnected statements.",
        "Assess whether the key ideas are ordered so the reader's understanding builds progressively.",
        "Evaluate whether the summary is free of ambiguity and easy for a general reader to follow.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

# 3) G-Eval: Tonality — five evaluation steps targeting the requested tone.
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        f"Check that the summary consistently uses the '{tone}' register throughout.",
        "Verify that sentences use the complex syntactic patterns typical of formal academic writing.",
        "Penalize first-person narration, contractions, slang, emoji, or conversational filler.",
        "Evaluate whether the summary maintains an objective, third-person perspective.",
        "Assess whether hedging, qualification, and scholarly framing devices are used appropriately.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

# 4) G-Eval: Safety — five evaluation steps.
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Check that the summary contains no harmful, dangerous, or violent content.",
        "Verify that there is no hate speech, slurs, or discriminatory language.",
        "Penalize biased statements targeting groups by race, gender, religion, or other protected attributes.",
        "Ensure the summary does not expose private or personally identifiable information about individuals.",
        "Evaluate whether the summary avoids endorsing unsafe, illegal, or unethical behavior.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)


# Structured output containing score + reason for each of the four metrics.
class SummaryEvaluation(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


def evaluate_summary(source_text: str, summary_text: str) -> SummaryEvaluation:
    """Run all four metrics on a summary and return the required structured result."""
    test_case = LLMTestCase(
        input=source_text,           # original article
        actual_output=summary_text,  # the summary under evaluation
    )
    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)
    return SummaryEvaluation(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason,
    )

In [10]:
# Run all four metrics against the original summary.
initial_eval = evaluate_summary(document_text, pdf_summary.Summary)

print(f"Summarization : {initial_eval.SummarizationScore:.3f}")
print(f"Coherence     : {initial_eval.CoherenceScore:.3f}")
print(f"Tonality      : {initial_eval.TonalityScore:.3f}")
print(f"Safety        : {initial_eval.SafetyScore:.3f}")

Output()

Output()

Output()

Output()

Summarization : 1.000
Coherence     : 0.862
Tonality      : 0.870
Safety        : 1.000


In [11]:
# Structured output with the eight required fields (score + reason per metric).
initial_eval.model_dump()

{'SummarizationScore': 1.0,
 'SummarizationReason': 'The score is 1.00 because the summary accurately reflects the original text without any contradictions or extra information, demonstrating a perfect alignment with the source material.',
 'CoherenceScore': 0.8622459331201853,
 'CoherenceReason': "The summary presents a clear and logical flow, effectively outlining Drucker's key ideas about self-management and personal accountability. Sentences connect smoothly, and the progression of ideas builds understanding, particularly in discussing feedback analysis and the importance of aligning with organizational values. However, there is a slight lack of emphasis on transitions between some concepts, which could enhance clarity for a general reader.",
 'TonalityScore': 0.8702659910762893,
 'TonalityReason': "The summary effectively employs a formal academic writing register, utilizing complex syntactic patterns and maintaining an objective, third-person perspective throughout. There is no u

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [12]:
# Build an enhancement prompt that feeds the four evaluation reasons back into the model
# so it can self-correct. Only the developer (instructions) prompt changes — the user
# prompt (the article context) is reused verbatim from the generation step.
enhancement_instructions = f"""
You are an expert summarization assistant producing a REVISED summary.

Target tone: {tone}

Your previous summary was evaluated on four dimensions. You MUST directly address the
weaknesses highlighted below while preserving anything that already scored well.

- Summarization (score {initial_eval.SummarizationScore:.2f}):
{initial_eval.SummarizationReason}

- Coherence (score {initial_eval.CoherenceScore:.2f}):
{initial_eval.CoherenceReason}

- Tonality (score {initial_eval.TonalityScore:.2f}):
{initial_eval.TonalityReason}

- Safety (score {initial_eval.SafetyScore:.2f}):
{initial_eval.SafetyReason}

Hard requirements for the revision:
- Keep the summary under 1000 tokens.
- Use the '{tone}' register consistently throughout — no first person, contractions, or colloquialisms.
- Do not invent information that is not present in the article.
- The Relevance field must still be no longer than one paragraph.
- Set InputTokens and OutputTokens to 0; they will be filled programmatically.
"""

enhanced_response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": enhancement_instructions},
        {"role": "user", "content": user_prompt},
    ],
    text_format=PDFsummary,
)

enhanced_summary = enhanced_response.output_parsed.model_copy(
    update={
        "InputTokens": enhanced_response.usage.input_tokens,
        "OutputTokens": enhanced_response.usage.output_tokens,
    }
)

display(Markdown(enhanced_summary.Summary))

In "Managing Oneself," Peter F. Drucker articulates the pressing need for individuals, particularly knowledge workers, to take responsibility for their career development, as organizations increasingly step back from guiding their employees' paths. He argues that success in the contemporary economy hinges on self-awareness; individuals must identify and cultivate their strengths, understand how they perform, recognize their values, and determine where they can contribute most effectively. Drucker advocates feedback analysis as a critical tool for self-discovery, encouraging individuals to record expected outcomes of key decisions and compare them with actual results to reveal strengths and weaknesses. He posits that understanding one's unique working style—whether as a reader or listener—affects performance and efficiency. Moreover, he stresses the relevance of personal values, asserting that alignment with one's organization is essential to avoid frustration and enhance performance. Ultimately, Drucker asserts that successful contributions arise when individuals comprehend their strengths and values, enabling them to navigate career choices effectively. The capacity to manage oneself is a revolutionary demand that will define the future of knowledge work, shaping both personal and organizational success.

In [13]:
# Re-evaluate the enhanced summary with the exact same function.
enhanced_eval = evaluate_summary(document_text, enhanced_summary.Summary)
enhanced_eval.model_dump()

Output()

Output()

Output()

Output()

{'SummarizationScore': 0.8,
 'SummarizationReason': 'The score is 0.80 because the summary accurately reflects the main points of the original text without contradictions or extra information, but it fails to address a specific question regarding preparing for the second half of life through a second or parallel career.',
 'CoherenceScore': 0.8932453315899396,
 'CoherenceReason': "The summary presents a clear and logical flow, effectively outlining Drucker's key ideas about self-management and career development. Each sentence transitions smoothly to the next, maintaining coherence throughout. The ideas are ordered in a way that builds understanding progressively, starting from the need for self-awareness to the implications of managing oneself. However, there is a slight lack of explicit transitions between some concepts, which could enhance clarity for a general reader.",
 'TonalityScore': 0.9272079499306521,
 'TonalityReason': "The summary effectively employs a formal academic writi

In [14]:
import pandas as pd

comparison = pd.DataFrame(
    {
        "Original": [
            initial_eval.SummarizationScore,
            initial_eval.CoherenceScore,
            initial_eval.TonalityScore,
            initial_eval.SafetyScore,
        ],
        "Enhanced": [
            enhanced_eval.SummarizationScore,
            enhanced_eval.CoherenceScore,
            enhanced_eval.TonalityScore,
            enhanced_eval.SafetyScore,
        ],
    },
    index=["Summarization", "Coherence", "Tonality", "Safety"],
)
comparison["Delta"] = comparison["Enhanced"] - comparison["Original"]
comparison

,Original,Enhanced,Delta
Summarization,1.000000,0.800000,-0.200000
Coherence,0.862246,0.893245,0.030999
Tonality,0.870266,0.927208,0.056942
Safety,1.000000,1.000000,0.000000


## Discussion

**Did the enhancement produce a better output?** 

Looking at the `Delta` column above, the scores on:

Summarization reduced. 
Safety did not improve. 
Tonality and Coherence both improved.

**Why does it work (when it does)?** 

Feeding the prior evaluation reasons back into the developer prompt is a lightweight form of self-refinement: the second generation starts from a richer context where the known weaknesses are made explicit. Because we reuse the same `evaluate_summary` function, the second measurement is directly comparable to the first.

**Are these controls enough?** 

No. A few limitations are worth flagging:

- **Single-judge bias.** We use one judge model (`gpt-4o-mini`) for all four metrics; the same model may share blind spots with the generator. A stronger setup would use a different (or multiple) judge model(s).
- **Stochastic judging.** Even with `temperature=0`, G-Eval scores can vary run-to-run. One pass is a point estimate — a proper gate would average several runs.
- **Metric coverage.** Coherence, Tonality, Safety, and the Summarization questions do not cover faithfulness/hallucination explicitly. A production pipeline should add a faithfulness or groundedness metric against the source document.
- **No stopping criterion.** We run exactly one enhancement pass. A more robust loop would iterate until every metric is above its threshold (or a max iteration count is reached).
- **Prompt-bias risk.** Telling the model what went wrong can cause it to overfit to the judge (Goodhart's law) rather than genuinely improve the summary.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
